In [5]:
import os
import sys
import torch
import transformers

sys.path.append(os.path.dirname(os.getcwd()))

from transformers import AutoTokenizer, GenerationConfig

from model.configuration_sophie0 import Sophie0Config
from model.modeling_sophie0 import Sophie0ForCausalLM

base_path = os.path.dirname(os.getcwd())
tokenizer_path = os.path.join(base_path, "model/tokenizer")
model_path = os.path.join(base_path, "result/dpo/final_baai/pytorch_model.bin")

tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True, trust_remote_code=True, local_files_only=True)
model = Sophie0ForCausalLM(Sophie0Config())

state_dict = torch.load(model_path, map_location='cpu', weights_only=True)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [6]:
device = "cuda:0"
dtype = torch.bfloat16
model = model.to(dtype=dtype, device=device)

In [7]:
generate_config = GenerationConfig(
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_new_tokens=256,
    do_sample=True,
    top_k=20,
    top_p=0.8,
    temperature=0.8,
    num_beams=1,
    repeat_penalty=1.2,
    use_cache=True
)

prompt = [
    "<s><user>请问你是由谁训练研发的呢？</s>\n<s><bot>",
    "<s><user>能否解释一下Transformer架构呢？</s>\n<s><bot>",
    "<s><user>Could you please give a C++ example for quick sort?</s>\n<s><bot>",
    "<s><user>能否介绍一下中国的首都呢？</s>\n<s><bot>",
    "<s><user>Could you please tell me a joke about the weather?</s>\n<s><bot>"
]
input_ids = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left").input_ids.to(device)

In [8]:
with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(input_ids, generate_config)

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <s><user>请问你是由谁训练研发的呢？</s>
<s><bot>相通看来owingvis看来rievedoweredrieved experience Daily Hello.ex deal refer看来看来涉猎 Hello experiencevis看来 Dailyoltvis edition editionowered看来 referoweredowingvis edition edition lines Daily lines listen Images listen listenvisthink edition lines.exowered errors saves saves experienceowered experience edition errors Daily refer saves Hello errors experienceowingoweredvisler Hello errorsowingowing experience.ex Hello Hello看来owing editionrieved lines Hello看来 edition listenvisoltvis禁 editionowered experiencerieved看来涉猎.ex琴 lines listen.ex涉猎visler看来.ex在海外.ex edition linesowered看来.ex看来 experience看来看来rieved edition涉猎 Daily lines edition.ex experience saves listen.ex edition linesvisvisvisler Daily琴相通涉猎.ex相通 listen Hello.ex琴vis edition涉猎rieved Hello listen涉猎相通涉猎rieved涉猎olt experienceowered experience涉猎 saves linesrieved lines涉猎 Hello涉猎.exrieved看来rievedowing lines看来rieved Hello errors listen禁涉猎 experience涉猎 Dailyoweredrieved涉猎琴 listenoweredvis errorsvis lines看来owere